In [1]:
import os
import time
import uuid
import chromadb
from groq import Groq
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
from datetime import datetime

load_dotenv()

print("Day 14 - Complete RAG Pipeline")
print("All imports successful")

# Initialize core components
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Groq client: ready")
print(f"Embedder: ready")

Day 14 - Complete RAG Pipeline
All imports successful
Groq client: ready
Embedder: ready


In [ ]:
@dataclass
class RAGResponse:
    """Structured response from the RAG pipeline"""
    query: str
    answer: str
    retrieved_chunks: List[Dict]
    latency_ms: float
    total_tokens: int
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    
    def display(self):
        print(f"Query: {self.query}")
        print(f"Answer: {self.answer}")
        print(f"Sources used: {len(self.retrieved_chunks)}")
        print(f"Latency: {self.latency_ms:.0f}ms")
        print(f"Tokens: {self.total_tokens}")


class EnterpriseRAGPipeline:
    """
    Complete Enterprise RAG Pipeline.
    Connects hybrid search + prompt engineering + LLM generation.
    This is the core of your resume project.
    """

    def __init__(
        self,
        org_id: str,
        groq_client: Groq,
        embedder: SentenceTransformer,
        persist_path: str = "./rag_pipeline_db",
        model: str = "llama-3.1-8b-instant",
        temperature: float=0.1,
        n_results: int =3
    ):
        self.org_id = org_id
        self.client = groq_client
        self.embedder = embedder
        self.model = model
        self.temperature = temperature
        self.n_results = n_results

        #ChromaDB
        self.chroma_client = chromadb.PersistentClient(path=persist_path)
        self.collection = self.chroma_client.get_or_create_collection(
            name=f"org_{org_id}"
        )

        # BM25
        self.bm25 = None
        self.bm25_corpus = []
        self.bm25_ids = []
        self.bm25_metadatas =[]

        # Conversation memory
        self.conversation_history = []

        # System prompt
        self.system_prompt = """You are an Enterprise RAG assistant.
Answer questions based ONLY on the provided document chunks.
Always cite which chunl you used (e.g 'According to chunk 1...').
If the answer is not in the chunks say exaxtly:
'I cannot find this information in the provided documents.'
Never make up information. Be concise and professional."""

        print(f"[{org_id}] Pipeline initialized")

    def ingest(
            self,
            texts: List[str],
            metadatas: List[Dict],
            ids: Optional[List[str]] = None
    )-> None:
        """Ingest documents into the pipeline"""
        if ids is None:
            ids = [str(uuid.uuid4())[:8] for _ in texts]

        # Add to ChromaDB
        embeddings = self.embedder.encode(texts).tolist()
        self.collection.add(
            ids=ids,
            embeddings = embeddings,
            documents =texts,
            metadatas = metadatas
        )

        # Add to BM25
        self.bm25_corpus.extend(texts)
        self.bm25_ids.extend(ids)
        self.bm25_metadatas.extend(metadatas)
        tokenized = [doc.lower().split() for doc in self.bm25_corpus]
        self.bm25 = BM25Okapi(tokenized)

        print(f"[{self.org_id}] Ingested {len(texts)} docs | Total: {len(self.bm25_corpus)}")


    def _vector_search(self, query: str, n: int) -> List[Tuple[str, float]]:
        query_embedding = self.embedder.encode(query).tolist()
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=min(n, self.collection.count())
        )
        return [
            (results['ids'][0][i], 1 -results['distances'][0][i])
            for i in range(len(results['ids'][0]))
        ]
    
    def _bm25_search(self, query: str, n:int) -> List[Tuple[str, float]]:
        if self.bm25 is None:
            return []
        scores = self.bm25.get_scores(query.lower().split())
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:n]
        return [(self.bm25_ids[idx], score) for idx, score in ranked]
    
    def _rrf(
        self,
        result_sets: List[List[Tuple[str, float]]],
        k: int = 60
    ) -> List[Tuple[str, float]]:
        rrf_scores = {}
        for result_set in result_sets:
            for rank, (doc_id, _) in enumerate(result_set):
                rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0/ (k+ rank+1)
        return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    def _retrieve(self, query: str)-> List[Dict]:
        """Hybrid retrieval - BM25 + vector + RRf"""
        _vector_results = self._vector_search(query, self.n_results*2)
        bm25_results = self._bm25_search(query, self.n_results*2)
        mergeg
        
        


SyntaxError: expected ':' (3067587754.py, line 26)